# Dunnhumby M2 선택 checkpoint: 가격축 제거 진단 (학습 없음)

완료된 seed 43, 선택 225 epoch checkpoint만 불러 `full / 가격축 제거 / N·V 제거 / ID-only`를 같은 신규상품 평가조건에서 재채점합니다. optimizer·추가 epoch·checkpoint 재선택·test·holdout은 없습니다. `ID-only`는 M2와 함께 학습된 ID이므로 M1이 아니라 원인분해용 view입니다.


In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil
from google.colab import drive, files
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_seed43_v3/reports/result.json'
TRAIN_OUT = ROOT/'results_v3_dunnhumby_m2_support_backoff_price_seed43_v1'
DIAG_OUT = ROOT/'results_v3_dunnhumby_m2_support_backoff_price_seed43_price_diagnostic_v1'
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not REPORT.is_file() or not (TRAIN_OUT/'reports/result.json').is_file():
        raise RuntimeError('기준 M1 또는 완료 M2 결과가 없습니다.')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결 실패. 이 셀부터 다시 실행하세요. 학습은 없습니다.') from exc
SOURCE_COMMIT = '417393dfa78493a4fe5170e52f88ceccf431ece8'
REPO = Path('/content/clv-m2-price-diagnostic-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
import clv_m2_support_backoff_price_checkpoint_diagnostic as diagnostic
print('source:', SOURCE_COMMIT, '| training:', TRAIN_OUT, '| diagnostic:', DIAG_OUT)


In [ ]:
context = diagnostic.prepare(REPORT, TRAIN_OUT, DIAG_OUT)
print(json.dumps({
    'code_version': diagnostic.VERSION,
    'selected_epoch': context['arm']['selected_epoch'],
    'views': diagnostic.VIEWS,
    'new_fit_count': 0, 'final_test': False, 'holdout': False
}, ensure_ascii=False, indent=2))


In [ ]:
import torch
assert torch.cuda.is_available(), '재채점 속도를 위해 GPU 런타임을 사용하세요.'
paths = diagnostic.run(context)
print(json.dumps(paths, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
absolute = pd.read_csv(paths['absolute']).set_index('view')
comparison = pd.read_csv(paths['comparison'])
movement = pd.read_csv(paths['movement_summary'])
report = json.loads(Path(paths['json']).read_text())
focus = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
         'price_purchase_amount_weighted_hit@10','vndcg@10',
         'price_purchase_amount_weighted_hit@20','vndcg@20',
         'price_purchase_amount_weighted_hit@50','vndcg@50']
print('1) 전체 절대지표')
display(absolute[focus].T)
print('2) 가격축 제거 vs full, 그리고 각 view vs M1')
display(comparison[(comparison.metric.isin(focus)) & (
    ((comparison.view == 'without_price') & (comparison.reference == 'full')) |
    (comparison.reference == 'm1'))].reset_index(drop=True))
print('3) 추천 이동 요약')
display(movement)
print('4) 판독')
print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
archive = shutil.make_archive('/content/m2_support_backoff_price_checkpoint_diagnostic', 'zip', DIAG_OUT)
files.download(archive)
